# Face-Car — treinamento no Google Colab

Este notebook executa a geração de dados sintéticos, o treinamento e a avaliação já existentes no projeto. Os resultados demonstram o funcionamento do fluxo com dados sintéticos; **não são uma validação com motoristas reais**.

A branch `feature/treinamento-avaliacao` precisa estar publicada no GitHub para que a célula de preparação consiga cloná-la. Nenhum dataset é baixado automaticamente.

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib joblib

## Preparar o código

Se o projeto já estiver disponível no ambiente, a célula o reutiliza. Caso contrário, clona a branch de treinamento do GitHub.

In [ ]:
import subprocess
from pathlib import Path

URL_REPOSITORIO = "https://github.com/dario-bastos-dev/face-car.git"
BRANCH = "feature/treinamento-avaliacao"
REPOSITORIO = Path("/content/face-car")
SCRIPT_GERACAO = Path.cwd() / "treinamento" / "preparar_dados.py"

if not SCRIPT_GERACAO.is_file():
    if REPOSITORIO.exists() and not (REPOSITORIO / "treinamento" / "preparar_dados.py").is_file():
        raise FileExistsError(f"O diretório {REPOSITORIO} existe, mas não contém o módulo de treinamento esperado.")
    if not REPOSITORIO.exists():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, "--single-branch", URL_REPOSITORIO, str(REPOSITORIO)],
            check=True,
        )
    PASTA_TREINAMENTO = REPOSITORIO / "treinamento"
else:
    PASTA_TREINAMENTO = SCRIPT_GERACAO.parent

print(f"Código de treinamento: {PASTA_TREINAMENTO.resolve()}")

## Gerar dados, treinar e avaliar

As chamadas usam os próprios caminhos dos scripts, então os arquivos de entrada e saída ficam dentro de `treinamento/`.

In [ ]:
import sys

for nome_script in ("preparar_dados.py", "treinar.py", "avaliar.py"):
    caminho_script = PASTA_TREINAMENTO / nome_script
    subprocess.run([sys.executable, str(caminho_script)], cwd=PASTA_TREINAMENTO, check=True)

In [ ]:
import pandas as pd

previsoes = pd.read_csv(PASTA_TREINAMENTO / "resultados" / "previsoes.csv")
print(previsoes.head().to_string(index=False))
print(f"Amostras avaliadas: {len(previsoes)}")
print("Atenção: estas previsões usam dados sintéticos, não dados de motoristas reais.")

## Salvar resultados no Google Drive (opcional)

Altere `SALVAR_NO_DRIVE` para `True` para montar o Drive e copiar o CSV sintético, o modelo, as previsões e os gráficos para `Meu Drive/face-car/treinamento/`.

In [ ]:
SALVAR_NO_DRIVE = False

if SALVAR_NO_DRIVE:
    import shutil
    from google.colab import drive

    drive.mount("/content/drive")
    destino = Path("/content/drive/MyDrive/face-car/treinamento")
    arquivos = (
        PASTA_TREINAMENTO / "dados" / "dados_sinteticos.csv",
        PASTA_TREINAMENTO / "modelos" / "modelo_sonolencia.joblib",
        PASTA_TREINAMENTO / "resultados" / "previsoes.csv",
        PASTA_TREINAMENTO / "resultados" / "metricas.png",
        PASTA_TREINAMENTO / "resultados" / "matriz_confusao.png",
    )
    destino.mkdir(parents=True, exist_ok=True)
    for arquivo in arquivos:
        shutil.copy2(arquivo, destino / arquivo.name)
    print(f"Arquivos copiados para {destino}")
else:
    print("Salvamento no Drive desativado; os arquivos permanecem no ambiente atual do Colab.")

## Preparação futura de datasets reais

Datasets como UTA-RLDD, NTHU Driver Drowsiness Detection Dataset e YawDD não são baixados por este notebook. Para usá-los, obtenha os arquivos respeitando os termos de cada dataset e integre posteriormente o extrator MediaPipe do projeto.

O treinamento espera um CSV com `motorista_id`, `ear`, `perclos`, `boca` e `sonolento`. As saídas atuais do extrator por quadro (`ear_esq`, `ear_dir`, `abertura_boca` e outras) precisam ser transformadas em janelas rotuladas; `perclos` exige uma definição temporal explícita e `sonolento` exige rótulos de referência. Consulte [README do módulo de treinamento](README.md) para detalhes. Não use métricas fictícias nem apresente os resultados sintéticos como validação real.